<a href="https://colab.research.google.com/github/ANKIT-KANDULNA/CS318_DL-LAB/blob/main/Experiment-5/DL_Lab_exp5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installing an setup

In [3]:
!pip install kaggle

# Upload kaggle.json manually
from google.colab import files
files.upload()

# Setup Kaggle API
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


Downloading Dataset

In [4]:
!kaggle datasets download -d imbikramsaha/poems
!unzip poems.zip

Dataset URL: https://www.kaggle.com/datasets/imbikramsaha/poems
License(s): CC0-1.0
100% 57.6k/57.6k [00:00<00:00, 39.4MB/s]

Archive:  poems.zip
  inflating: poems-100.csv           


Loading Dataset

In [5]:
import os

# Check files
for root, dirs, files in os.walk('/content'):
    for file in files:
        if file.endswith('.txt'):
            print(os.path.join(root, file))

In [7]:
# Load poems text
file_path = '/content/poems-100.csv'  # adjust if path different

with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

print(text[:500])

text
"O my Luve's like a red, red rose
That’s newly sprung in June;
O my Luve's like the melodie
That’s sweetly play'd in tune.

As fair art thou, my bonnie lass,
So deep in luve am I:
And I will luve thee still, my dear,
Till a’ the seas gang dry:

Till a’ the seas gang dry, my dear,
And the rocks melt wi’ the sun:
I will luve thee still, my dear,
While the sands o’ life shall run.

And fare thee well, my only Luve
And fare thee well, a while!
And I will come again, my Luve,
Tho’ it were ten th


# Preprocessing

In [8]:
import numpy as np
import tensorflow as tf

# unique characters
chars = sorted(list(set(text)))
char2idx = {u:i for i,u in enumerate(chars)}
idx2char = np.array(chars)

text_as_int = np.array([char2idx[c] for c in text])

Create Sequences

In [9]:
seq_length = 100
examples_per_epoch = len(text)//(seq_length+1)

char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)

sequences = char_dataset.batch(seq_length+1, drop_remainder=True)

def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)

Batch with Shuffle

In [10]:
BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

# Build RNN Model

In [11]:
vocab_size = len(chars)
embedding_dim = 256
rnn_units = 512

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim),
    tf.keras.layers.SimpleRNN(rnn_units, return_sequences=True),
    tf.keras.layers.Dense(vocab_size)
])

def loss(labels, logits):
    return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

model.compile(optimizer='adam', loss=loss)

# Training the Model

In [12]:
EPOCHS = 10

history = model.fit(dataset, epochs=EPOCHS)

Epoch 1/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 14s 489ms/step - loss: 3.1936
Epoch 2/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 10s 482ms/step - loss: 2.6298
Epoch 3/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 9s 425ms/step - loss: 2.3754
Epoch 4/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 11s 463ms/step - loss: 2.2476
Epoch 5/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 11s 500ms/step - loss: 2.1623
Epoch 6/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 13s 584ms/step - loss: 2.1014
Epoch 7/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 17s 753ms/step - loss: 2.0492
Epoch 8/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 13s 423ms/step - loss: 2.0082
Epoch 9/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 10s 482ms/step - loss: 1.9676
Epoch 10/10
21/21 ━━━━━━━━━━━━━━━━━━━━ 10s 485ms/step - loss: 1.9339


# Generating text

In [13]:
def generate_text(model, start_string):
    num_generate = 500

    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)

    text_generated = []

    temperature = 1.0

    for i in range(num_generate):
        predictions = model(input_eval)
        predictions = tf.squeeze(predictions, 0)

        predictions = predictions / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1,0].numpy()

        input_eval = tf.expand_dims([predicted_id], 0)

        text_generated.append(idx2char[predicted_id])

    return start_string + ''.join(text_generated)

In [14]:
print(generate_text(model, start_string="Love"))

LoverithinGinghe.-g s ckene senory gq..""?
 l stheJ1f g, Bumake my t, h,
Anve?
"Of bSplidonive“Qky Jk, G!Ry My ThGzoX…Lat gsh
BZf OZ
 Joy-YDichecoulas, t Agre.
Te;
I wyowhthedades,
 axDJ‘‘“Durgigas d d: k I w—RKy t blin: g! s, weB?"Than Lke deFy
The s,
Thexoringing pathe, ffrexbPawadst gallofrdine,
Le s d ca Y-Ongosen-by,
S
Mancourona Wore jo—I Angh'd My Nrvee avew cout,
WhtI Wh My Es,
Anen—Thetheraly linoo."n,
Wus, itY?
Nowh'dQ-Gby fincot E–
 t Thathat angonded fwithin'de the anowit!“And,
 tl,
Ofod
